# 模型初始測試

In [65]:
# model_name = "llama-3.2-3b-instruct" #3B模型太笨
# model_name = "meta-llama-3.1-8b-instruct"
model_name = "gemma-3-12b-it"

In [14]:
from openai import OpenAI
from copy import copy, deepcopy
import json
# Point to the local server
# client = OpenAI(base_url="http://192.168.31.200:1234/v1", api_key="lm-studio")
client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")

completion = client.chat.completions.create(
  model=model_name,
  messages=[
    {"role": "system", "content": "簡單的回答使用者的問題"},
    {"role": "user", "content": "機器學習"}
  ],
  temperature=0.7,
)

print(completion.choices[0].message.content)
print('===============')
print(completion.choices[0])

機器學習是人工智慧的一個分支，旨在教導計算機如何從數據中學習和改善其預測或決策能力。它的主要目的是讓機器可以從資料中自動提取模式、特徵和規律，然後使用這些知識來進行預測、分類、回歸等分析。

常見的機器學習算法包括：

1. 線性回歸：用於連續值預測
2. 梯度提升決策樹（GBDT）：用於分類和回歸
3. 支援向量機（SVM）：用於分類
4. 隨機森林：用於分類和回歸
5. 人工神經網絡（ANN）：用於預測和分類

機器學習的應用包括：

1. 自動化：用於自動化各種業務流程
2. 預測分析：用於預測未來事件或趨勢
3. 個人化推薦：用於提供針對個別客戶的商品或服務推薦
4. 文本分析：用於分析和理解文本資料

機器學習的優點包括：

1. 自動化：可以自動完成多項任務
2. 快速：可以快速處理大量數據
3. 精確度高：可以提供較為準確的預測或決策結果

然而，機器學習也有一些缺點，如：

1. 需要大量數據：需要足夠多樣化和豐富的數據才能得到可靠的模型
2. 可能出現過度擬合：如果模型太複雜，它可能會過度擬合-training data，導致在測試資料上的表現不佳。
3. 需要高昂的計算能力和儲存空間：機器學習需要大量的計算能力和儲存空間來處理和儲存數據。
Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='機器學習是人工智慧的一個分支，旨在教導計算機如何從數據中學習和改善其預測或決策能力。它的主要目的是讓機器可以從資料中自動提取模式、特徵和規律，然後使用這些知識來進行預測、分類、回歸等分析。\n\n常見的機器學習算法包括：\n\n1. 線性回歸：用於連續值預測\n2. 梯度提升決策樹（GBDT）：用於分類和回歸\n3. 支援向量機（SVM）：用於分類\n4. 隨機森林：用於分類和回歸\n5. 人工神經網絡（ANN）：用於預測和分類\n\n機器學習的應用包括：\n\n1. 自動化：用於自動化各種業務流程\n2. 預測分析：用於預測未來事件或趨勢\n3. 個人化推薦：用於提供針對個別客戶的商品或服務推薦\n4. 文本分析：用於分析和理解文本資料\n\n機器學習的優點包括：\n\n1. 自動化：可以自動完成多項任務\n2. 快速：可以快速

# 開始測試

In [ ]:
! pip install geopy

In [29]:
from geopy.geocoders import Nominatim

def get_coordinates(city_name):
    geolocator = Nominatim(user_agent="clement@example.com")
    location = geolocator.geocode(city_name)
    if location:
        return (location.latitude, location.longitude)
    else:
        return None


# get_coordinates("Taipei")
get_coordinates("台北")

(25.0375198, 121.5636796)

In [30]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

get_weather(25.047554, 121.5170503) #台北市溫度

34.1

In [7]:
tools = [     
    {
        "type": "function",
        "function": {
            "name": "get_coordinates",
            "description": "取得城市的GPS座標",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": { "type": "string", "description": "城市名稱" }
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "取得溫度值",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": { "type": "number", "description": "GPS經度"},
                    "longitude": { "type": "number", "description": "GPS緯度" }
                }
            },
        },
    }
] 

In [8]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")

system_prompt = '''
你是一個個人助理。你的任務是簡潔的回答使用者的問題。
當你需要使用工具時，可以直接選擇使用，不需要詢問使用者。
'''

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "紐約今天適合出門玩嗎？",}
]

completion = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools
)
print(completion.choices[0])

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='329378299', function=Function(arguments='{"city_name":"New York"}', name='get_coordinates'), type='function')]))


In [9]:
completion.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='329378299', function=Function(arguments='{"city_name":"New York"}', name='get_coordinates'), type='function')]

In [11]:
import json
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_coordinates(**args)
result

(40.7127281, -74.0060152)

In [12]:
messages.append(completion.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})
completion = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
)
completion.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='851949517', function=Function(arguments='{"latitude":40.7127281,"longitude":-74.0060152}', name='get_weather'), type='function')]))

In [13]:
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(**args)
result

17.5

In [14]:
messages.append(completion.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})
completion = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
)
completion.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='紐約今天溫度是攝氏17.5度，適合出門玩。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[]))

# 將流程串起來

In [42]:
from geopy.geocoders import Nominatim

def get_coordinates(city_name):
    geolocator = Nominatim(user_agent="clement@example.com")
    location = geolocator.geocode(city_name)
    if location:
        return (location.latitude, location.longitude)
    else:
        return None

import requests
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


tools = [     
    {
        "type": "function",
        "function": {
            "name": "get_coordinates",
            "description": "取得城市的GPS座標",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": { "type": "string", "description": "城市名稱" }
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "取得溫度值",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": { "type": "number", "description": "GPS經度"},
                    "longitude": { "type": "number", "description": "GPS緯度" }
                }
            },
        },
    }
] 

In [53]:
class weather_bot:
    def __init__(self, model_name, tools):
        self.client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
        self.model_name = model_name
        self.tools = tools
        self.messages = [
            # {"role": "system", "content": "你是一個商城的購物助手，可以幫助會員尋找相關商品並放入購物車裡面。"},#這是原始的
            {"role": "system", "content": '''
            你是一個個人助理。你的任務是簡潔的回答使用者的問題。
            當你需要使用工具時，可以直接選擇使用，不需要詢問使用者。
            使用繁體中文來回答。
            '''
            },
        ]


    def call_function(self, tool_call):
        # print(tool_call)
        # print(tool_call.function)
        # print(tool_call.function.arguments)
        args = json.loads(tool_call.function.arguments)
        try:
            print(f'call: {tool_call.function.name}({args})')
            result = globals()[tool_call.function.name](**args)
            print(f'return : {result}')
            return result
        except:
            print(f'call: {tool_call.function.name}() 失敗')
            return f"Call {tool_call.function.name} 失敗"
    
    # def call_function(self, tool_call):
    #     args = json.loads(tool_call.function.arguments)
    #     if tool_call.function.name == 'get_weather':
    #         return get_weather(**args)
    #     elif tool_call.function.name == 'get_coordinates':
    #         return get_coordinates(**args)
    #     else:
    #         return None
        
    def chat(self, text):
        history = ''
        self.messages.append({"role": "user", "content": text})
        
        while True:
            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=self.messages,
                tools=self.tools
            )            

            is_tools_call = False
            for choice in completion.choices:
                # print(choice)
                self.messages.append(choice.message)
                if choice.message.tool_calls:
                    is_tools_call = True
                    for tool_call in choice.message.tool_calls:
                        result = self.call_function(tool_call)

                        history = f"{history}[我執行了 {tool_call.function.name}]\n"
                        self.messages.append({                               # append result message
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": str(result)
                        })                        
                else:
                    history = history + choice.message.content

            if is_tools_call == False:
                return history
    

In [66]:
bot = weather_bot(model_name, tools)

In [67]:
print(bot.chat("你好"))

您好！有什麼我可以幫您的嗎？



In [68]:
print(bot.chat("我想要知道雪梨今天適合出門玩嗎"))

call: get_coordinates({'city_name': '雪梨'})
return : (-33.8698439, 151.2082848)
call: get_weather({'latitude': -33.8698439, 'longitude': 151.2082848})
return : 19.2
[我執行了 get_coordinates]
[我執行了 get_weather]
雪梨今天溫度約為 19.2 度，適合出門玩。


In [69]:
print(bot.chat("這樣的溫度會很冷嗎"))

這取決於您對溫度的感受。一般來說，19.2度在台灣可能覺得涼爽，但如果習慣了更熱的氣候，可能會覺得有點冷。建議您可以穿著薄外套或長袖衣物出門。


### 使用Gradio做介面

In [29]:
import gradio as gr
from google import genai

bot = weather_bot(API_KEY, model_name, config)
def chat_function(message, history):
    response = bot.chat(message)
    return response

demo = gr.ChatInterface(chat_function, type="messages", autofocus=False)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
